## Operator Distillation

This notebook focuses on study of replacement operator training. Operator replacement is performed by training a replacement using knowledge distillation utilizing calibration data pairs.

problem:

Initial experiments showed bad generalization of selected operators. We have achieved NMSE around 0.5 on validation data which is not usable as a proper fit, ideally we want to achieve $NMSE \leq 0.05$, this we consider a good operator fit.

### Axes of study

This notebook studies how to improve the generalization of local activation distillation while keeping the replacement architecture fixed.

1. **Calibration data** — number and diversity of activation pairs.
2. **Initialization** — random initialization versus teacher-derived weights.
3. **Optimization** — learning rate, schedule, batch size, optimizer, and update budget.
4. **Regularization and loss** — weight decay, early stopping, and regression objective.
5. **Transfer across layers** — whether the selected recipe generalizes beyond the reference block.

Configurations are compared using held-out NMSE, cosine similarity, and the train–validation gap.

### setup

In [ ]:
from dataclasses import asdict, replace
from datetime import datetime, timezone
import json
from math import ceil
from pathlib import Path
import sys

import matplotlib.pyplot as plt
import pandas as pd
import seaborn as sns
import torch

In [ ]:
def find_project_root(start):
    start = start.resolve()
    for candidate in (start, *start.parents):
        if (candidate / 'src' / 'mlp_replacement').is_dir():
            return candidate
    raise RuntimeError('Could not locate the repository root')

In [ ]:
PROJECT_ROOT = find_project_root(Path.cwd())
SRC = PROJECT_ROOT / 'src'
if str(SRC) not in sys.path:
    sys.path.insert(0, str(SRC))

from mlp_replacement.capture import ActivationPairs, collect_module_io
from mlp_replacement.config import DataConfig, ModelConfig, OperatorConfig
from mlp_replacement.data import build_data_loaders
from mlp_replacement.evaluation.operator import evaluate_operator
from mlp_replacement.model import get_mlp_block, load_model_and_tokenizer
from mlp_replacement.operators import GatedMLPReplacement, fit_operator

In [ ]:
SEED = 21
torch.manual_seed(SEED)
sns.set_theme(style='whitegrid', context='notebook')

In [ ]:
EXECUTION_MODE = 'run'
ARTIFACT_SCHEMA_VERSION = 3

TARGET_LAYER = 11
SWIGLU_WIDTH_RATIO = 0.50

BASELINE_EXPERIMENTS_ARTIFACT_PATH = (
    PROJECT_ROOT
    / 'data'
    / 'results'
    / 'notebook-block-study'
    / 'baseline-experiments.json'
)
OUTPUT_PATH = (
    PROJECT_ROOT
    / 'data'
    / 'results'
    / 'notebook-block-study'
    / 'operator-distillation-v2.json'
)
RUN_FROM_SCRATCH = EXECUTION_MODE == 'run'
loaded_artifact = (
    None
    if RUN_FROM_SCRATCH
    else json.loads(OUTPUT_PATH.read_text(encoding='utf-8'))
)
if (
    not RUN_FROM_SCRATCH
    and loaded_artifact.get('schema_version') != ARTIFACT_SCHEMA_VERSION
):
    raise ValueError(
        'The saved artifact uses the legacy methodology. '
        "Set EXECUTION_MODE = 'run' to regenerate it."
    )
if not RUN_FROM_SCRATCH:
    SEED = loaded_artifact['configuration']['seed']
    TARGET_LAYER = loaded_artifact['configuration']['target_layer']
    SWIGLU_WIDTH_RATIO = loaded_artifact['configuration'][
        'swiglu_width_ratio'
    ]

In [ ]:
model_config = ModelConfig(
    model_id='HuggingFaceTB/SmolLM2-1.7B',
    device='auto',
    dtype='auto',
)

data_config = DataConfig(
    sequence_length=128,
    batch_size=2,
    num_calibration_batches=48,
    num_operator_validation_batches=24,
    num_recovery_batches=0,
    num_recovery_validation_batches=0,
    num_model_validation_batches=24,
    num_test_batches=0,
    seed=SEED,
)

baseline_training_config = OperatorConfig(
    kind='swiglu',
    intermediate_ratio=SWIGLU_WIDTH_RATIO,
    epochs=64,
    learning_rate=1e-3,
    batch_size=2048,
    weight_decay=0.0,
    scheduler='constant',
    early_stopping_patience=3,
    seed=SEED,
)

CALIBRATION_BATCH_BUDGETS = [48, 96, 192, 384]
CALIBRATION_MAX_EPOCHS = 64
if not RUN_FROM_SCRATCH:
    CALIBRATION_BATCH_BUDGETS = loaded_artifact['configuration'][
        'calibration_batch_budgets'
    ]
    CALIBRATION_MAX_EPOCHS = loaded_artifact['configuration'][
        'calibration_max_epochs'
    ]

In [ ]:
if RUN_FROM_SCRATCH:
    model, tokenizer = load_model_and_tokenizer(model_config)
    device = next(model.parameters()).device
else:
    model = tokenizer = device = None

In [ ]:
additional_calibration_batches = (
    max(CALIBRATION_BATCH_BUDGETS)
    - data_config.num_calibration_batches
)
if RUN_FROM_SCRATCH:
    # The unused recovery partition keeps extra calibration data
    # after the fixed operator-validation partition.
    calibration_study_data_config = replace(
        data_config,
        num_recovery_batches=additional_calibration_batches,
    )
    loaders = build_data_loaders(
        tokenizer,
        calibration_study_data_config,
        include_recovery=True,
    )
else:
    calibration_study_data_config = None
    loaders = None

In [ ]:
if RUN_FROM_SCRATCH:
    block = get_mlp_block(model, TARGET_LAYER)
    hidden_size = int(model.config.hidden_size)
    original_intermediate_width = int(
        block.module.up_proj.out_features
    )
    replacement_width = max(
        1,
        round(SWIGLU_WIDTH_RATIO * original_intermediate_width),
    )
else:
    artifact_configuration = loaded_artifact['configuration']
    TARGET_LAYER = artifact_configuration['target_layer']
    SWIGLU_WIDTH_RATIO = artifact_configuration[
        'swiglu_width_ratio'
    ]
    hidden_size = artifact_configuration['hidden_size']
    original_intermediate_width = artifact_configuration[
        'original_intermediate_width'
    ]
    replacement_width = artifact_configuration[
        'replacement_width'
    ]
    block = None

## Experiments

#### Original distillation setting

Recompute the controlled baseline with the corrected 4,096-wide SwiGLU operator. The first 48 calibration batches and following 24 validation batches reproduce the original data setting. Additional calibration samples are drawn only after this fixed validation partition and are used later.

In [ ]:
if RUN_FROM_SCRATCH:
    baseline_training_pairs = collect_module_io(
        model,
        block.path,
        loaders.calibration,
        data_config.num_calibration_batches,
        device,
    )
    calibration_validation_pairs = collect_module_io(
        model,
        block.path,
        loaders.operator_validation,
        data_config.num_operator_validation_batches,
        device,
    )
    additional_calibration_pairs = collect_module_io(
        model,
        block.path,
        loaders.recovery,
        additional_calibration_batches,
        device,
    )
    calibration_training_pool = ActivationPairs(
        inputs=torch.cat([
            baseline_training_pairs.inputs,
            additional_calibration_pairs.inputs,
        ]),
        targets=torch.cat([
            baseline_training_pairs.targets,
            additional_calibration_pairs.targets,
        ]),
    )

    calibration_tokens = baseline_training_pairs.inputs.shape[0]
    validation_tokens = calibration_validation_pairs.inputs.shape[0]
    baseline_steps_per_epoch = ceil(
        calibration_tokens / baseline_training_config.batch_size
    )

    torch.manual_seed(SEED)
    baseline_fit = fit_operator(
        GatedMLPReplacement(
            hidden_size, replacement_width, bias=False
        ),
        baseline_training_pairs,
        calibration_validation_pairs,
        baseline_training_config,
        device,
    )
    baseline_train_metrics = evaluate_operator(
        baseline_fit.module,
        baseline_training_pairs,
        device,
        baseline_training_config.batch_size,
    )
    baseline_validation_metrics = evaluate_operator(
        baseline_fit.module,
        calibration_validation_pairs,
        device,
        baseline_training_config.batch_size,
    )

    baseline_history_df = pd.DataFrame([
        asdict(epoch) for epoch in baseline_fit.history
    ])
    baseline_history_df['optimizer_updates'] = (
        baseline_history_df['epoch'] * baseline_steps_per_epoch
    )
    baseline_history_df['pair_presentations'] = (
        baseline_history_df['epoch'] * calibration_tokens
    )
    baseline_parameters = sum(
        parameter.numel()
        for parameter in baseline_fit.module.parameters()
    )

    baseline_config_df = pd.DataFrame([
        {'setting': 'model revision', 'value': getattr(model.config, '_commit_hash', None)},
        {'setting': 'target layer', 'value': TARGET_LAYER},
        {'setting': 'teacher intermediate width', 'value': original_intermediate_width},
        {'setting': 'replacement intermediate width', 'value': replacement_width},
        {'setting': 'SwiGLU width ratio', 'value': SWIGLU_WIDTH_RATIO},
        {'setting': 'parameters', 'value': baseline_parameters},
        {'setting': 'calibration pairs', 'value': calibration_tokens},
        {'setting': 'validation pairs', 'value': validation_tokens},
        {'setting': 'maximum epochs', 'value': baseline_training_config.epochs},
        {'setting': 'operator batch size', 'value': baseline_training_config.batch_size},
        {'setting': 'learning rate', 'value': baseline_training_config.learning_rate},
        {'setting': 'scheduler', 'value': baseline_training_config.scheduler},
        {'setting': 'weight decay', 'value': baseline_training_config.weight_decay},
        {'setting': 'early-stopping patience', 'value': baseline_training_config.early_stopping_patience},
        {'setting': 'seed', 'value': SEED},
    ])
    baseline_metrics_df = pd.DataFrame([{
        'name': 'swiglu_0.50_baseline',
        'parameters': baseline_parameters,
        'completed_epochs': len(baseline_fit.history),
        'best_epoch': baseline_fit.best_epoch,
        'best_optimizer_update': (
            baseline_fit.best_epoch * baseline_steps_per_epoch
        ),
        'selected_train_mse': baseline_train_metrics.mse,
        'local_mse': baseline_validation_metrics.mse,
        'local_relative_mse': baseline_validation_metrics.relative_mse,
        'local_cosine': baseline_validation_metrics.cosine_similarity,
        'local_r2': baseline_validation_metrics.r2,
        'local_norm_ratio': baseline_validation_metrics.norm_ratio,
        'median_token_relative_error': (
            baseline_validation_metrics.median_token_relative_error
        ),
        'p95_token_relative_error': (
            baseline_validation_metrics.p95_token_relative_error
        ),
    }])
    baseline_calibration_row = {
        'calibration_batches': data_config.num_calibration_batches,
        'calibration_tokens': calibration_tokens,
        'operator_batch_size': baseline_training_config.batch_size,
        'maximum_epochs': baseline_training_config.epochs,
        'completed_epochs': len(baseline_fit.history),
        'steps_per_epoch': baseline_steps_per_epoch,
        'completed_optimizer_updates': (
            len(baseline_fit.history) * baseline_steps_per_epoch
        ),
        'best_epoch': baseline_fit.best_epoch,
        'best_optimizer_update': (
            baseline_fit.best_epoch * baseline_steps_per_epoch
        ),
        'pair_presentations': (
            len(baseline_fit.history) * calibration_tokens
        ),
        'selected_train_mse': baseline_train_metrics.mse,
        'selected_validation_mse': baseline_validation_metrics.mse,
        'train_validation_gap': (
            baseline_validation_metrics.mse - baseline_train_metrics.mse
        ),
        'heldout_nmse': baseline_validation_metrics.relative_mse,
        'heldout_cosine': baseline_validation_metrics.cosine_similarity,
    }
else:
    baseline_training_pairs = None
    additional_calibration_pairs = None
    calibration_training_pool = None
    calibration_validation_pairs = None
    baseline_calibration_row = None
    original_results = loaded_artifact['results'][
        'original_distillation'
    ]
    baseline_config_df = pd.DataFrame(
        original_results['configuration']
    )
    baseline_metrics_df = pd.DataFrame(
        original_results['metrics']
    )
    baseline_history_df = pd.DataFrame(
        original_results['training_history']
    )
    calibration_tokens = loaded_artifact['configuration'][
        'original_calibration_tokens'
    ]

display(baseline_config_df)
display(baseline_metrics_df)

In [ ]:
baseline_history_long = baseline_history_df.melt(
    id_vars=['epoch', 'optimizer_updates'],
    value_vars=['train_mse', 'validation_mse'],
    var_name='split',
    value_name='mse',
)

plt.figure(figsize=(8, 4))
sns.lineplot(
    data=baseline_history_long,
    x='epoch',
    y='mse',
    hue='split',
    marker='o',
)
plt.axvline(
    int(baseline_metrics_df.iloc[0]['best_epoch']),
    color='black',
    linestyle='--',
    linewidth=1,
    label='selected checkpoint',
)
plt.title('Recomputed SwiGLU-0.50 baseline')
plt.xlabel('epoch')
plt.ylabel('MSE')
plt.legend()
plt.tight_layout()
plt.show()

In [ ]:
baseline_experiments_artifact = (
    json.loads(
        BASELINE_EXPERIMENTS_ARTIFACT_PATH.read_text(encoding='utf-8')
    )
    if RUN_FROM_SCRATCH
    else None
)
width_scaling_df = pd.DataFrame(
    baseline_experiments_artifact['results']['swiglu_width_scaling']
    if RUN_FROM_SCRATCH
    else loaded_artifact['results']['original_distillation'][
        'width_scaling'
    ]
).sort_values('width_ratio')

plt.figure(figsize=(7, 4))
sns.lineplot(
    data=width_scaling_df,
    x='width_ratio',
    y='local_relative_mse',
    marker='o',
)
plt.title('Architecture context: SwiGLU width scaling')
plt.xlabel('replacement width ratio')
plt.ylabel('held-out NMSE')
plt.tight_layout()
plt.show()

### Fitting improvements

#### Calibration data increase

Compare calibration sizes with the same batch size and a maximum of 64 epochs. This isolates the effect of adding unique activation pairs; larger datasets are intentionally allowed more optimizer updates. Saved outputs below remain from the legacy run until these cells are executed again.

In [ ]:
if RUN_FROM_SCRATCH:
    calibration_rows = [baseline_calibration_row]
    calibration_history_rows = [
        {'calibration_tokens': calibration_tokens, **row}
        for row in baseline_history_df.to_dict(orient='records')
    ]
else:
    calibration_rows = []
    calibration_history_rows = []

In [ ]:
tokens_per_capture_batch = (
    data_config.batch_size * data_config.sequence_length
)
for num_batches in (
    [
        value for value in CALIBRATION_BATCH_BUDGETS
        if value != data_config.num_calibration_batches
    ]
    if RUN_FROM_SCRATCH
    else []
):
    num_tokens = num_batches * tokens_per_capture_batch
    training_pairs = ActivationPairs(
        inputs=calibration_training_pool.inputs[:num_tokens],
        targets=calibration_training_pool.targets[:num_tokens],
    )
    steps_per_epoch = ceil(
        num_tokens / baseline_training_config.batch_size
    )
    training_config = replace(
        baseline_training_config,
        epochs=CALIBRATION_MAX_EPOCHS,
    )

    torch.manual_seed(SEED)
    fit = fit_operator(
        GatedMLPReplacement(
            hidden_size, replacement_width, bias=False
        ),
        training_pairs,
        calibration_validation_pairs,
        training_config,
        device,
    )
    train_metrics = evaluate_operator(
        fit.module,
        training_pairs,
        device,
        training_config.batch_size,
    )
    validation_metrics = evaluate_operator(
        fit.module,
        calibration_validation_pairs,
        device,
        training_config.batch_size,
    )

    calibration_rows.append({
        'calibration_batches': num_batches,
        'calibration_tokens': num_tokens,
        'operator_batch_size': training_config.batch_size,
        'maximum_epochs': training_config.epochs,
        'completed_epochs': len(fit.history),
        'steps_per_epoch': steps_per_epoch,
        'completed_optimizer_updates': (
            len(fit.history) * steps_per_epoch
        ),
        'best_epoch': fit.best_epoch,
        'best_optimizer_update': (
            fit.best_epoch * steps_per_epoch
        ),
        'pair_presentations': (
            len(fit.history) * num_tokens
        ),
        'selected_train_mse': train_metrics.mse,
        'selected_validation_mse': validation_metrics.mse,
        'train_validation_gap': (
            validation_metrics.mse - train_metrics.mse
        ),
        'heldout_nmse': validation_metrics.relative_mse,
        'heldout_cosine': (
            validation_metrics.cosine_similarity
        ),
    })
    for epoch in fit.history:
        calibration_history_rows.append({
            'calibration_tokens': num_tokens,
            'epoch': epoch.epoch,
            'optimizer_updates': epoch.epoch * steps_per_epoch,
            'pair_presentations': epoch.epoch * num_tokens,
            'train_mse': epoch.train_mse,
            'validation_mse': epoch.validation_mse,
        })

if RUN_FROM_SCRATCH:
    calibration_study_df = pd.DataFrame(calibration_rows)
    calibration_history_df = pd.DataFrame(
        calibration_history_rows
    )
else:
    calibration_results = loaded_artifact['results'][
        'calibration_data'
    ]
    calibration_study_df = pd.DataFrame(
        calibration_results['summary']
    )
    calibration_history_df = pd.DataFrame(
        calibration_results['training_history']
    )

In [ ]:
display(calibration_study_df)

In [ ]:
calibration_history_plot_df = calibration_history_df.copy()
calibration_history_plot_df['calibration budget'] = (
    calibration_history_plot_df['calibration_tokens']
    .map(lambda value: f'{value:,}')
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
sns.lineplot(
    data=calibration_history_plot_df,
    x='epoch',
    y='train_mse',
    hue='calibration budget',
    ax=axes[0],
)
sns.lineplot(
    data=calibration_history_plot_df,
    x='epoch',
    y='validation_mse',
    hue='calibration budget',
    ax=axes[1],
)
axes[0].set(title='Online training MSE', ylabel='MSE')
axes[1].set(title='Validation MSE', ylabel='MSE')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.lineplot(
    data=calibration_study_df,
    x='calibration_tokens',
    y='heldout_nmse',
    marker='o',
)
plt.title('Calibration data with up to 64 epochs')
plt.xlabel('calibration activation pairs')
plt.ylabel('held-out NMSE')
plt.tight_layout()
plt.show()

#### Teacher-derived initialization

The purpose of this section is to implement teacher-derived starting weight initialization. Instead of starting from scratch we can re-use some weights from the teacher itself for better local learning.

1) Random init - pick weights randomly from original
2) Informed init - using some sort of weight importance metric pick weights

1. Random init

2. Informed init

#### Training optimization

Screen operator batch sizes 256, 512, 1,024, and 2,048 on the largest calibration pool for 8 fixed epochs. All other fitting settings remain unchanged.

In [ ]:
OPERATOR_BATCH_SIZES = [256, 512, 1024, 2048]
TRAINING_OPTIMIZATION_EPOCHS = 8
TRAINING_OPTIMIZATION_SPECS = [
    {
        'configuration': f'batch_{batch_size}',
        'operator_batch_size': batch_size,
    }
    for batch_size in OPERATOR_BATCH_SIZES
]
if not RUN_FROM_SCRATCH:
    OPERATOR_BATCH_SIZES = loaded_artifact[
        'configuration'
    ]['operator_batch_sizes']
    TRAINING_OPTIMIZATION_EPOCHS = loaded_artifact[
        'configuration'
    ]['training_optimization_epochs']
    TRAINING_OPTIMIZATION_SPECS = loaded_artifact[
        'configuration'
    ]['training_optimization_specs']

In [ ]:
optimization_calibration_tokens = (
    max(CALIBRATION_BATCH_BUDGETS) * tokens_per_capture_batch
)
optimization_training_pairs = (
    ActivationPairs(
        inputs=calibration_training_pool.inputs[
            :optimization_calibration_tokens
        ],
        targets=calibration_training_pool.targets[
            :optimization_calibration_tokens
        ],
    )
    if RUN_FROM_SCRATCH
    else None
)
optimization_rows = []
optimization_history_rows = []

for spec in (
    TRAINING_OPTIMIZATION_SPECS if RUN_FROM_SCRATCH else []
):
    steps_per_epoch = ceil(
        optimization_calibration_tokens
        / spec['operator_batch_size']
    )
    training_config = replace(
        baseline_training_config,
        epochs=TRAINING_OPTIMIZATION_EPOCHS,
        batch_size=spec['operator_batch_size'],
        early_stopping_patience=None,
    )

    torch.manual_seed(SEED)
    fit = fit_operator(
        GatedMLPReplacement(
            hidden_size, replacement_width, bias=False
        ),
        optimization_training_pairs,
        calibration_validation_pairs,
        training_config,
        device,
    )
    train_metrics = evaluate_operator(
        fit.module,
        optimization_training_pairs,
        device,
        training_config.batch_size,
    )
    validation_metrics = evaluate_operator(
        fit.module,
        calibration_validation_pairs,
        device,
        training_config.batch_size,
    )

    optimization_rows.append({
        **spec,
        'calibration_tokens': optimization_calibration_tokens,
        'maximum_epochs': training_config.epochs,
        'completed_epochs': len(fit.history),
        'steps_per_epoch': steps_per_epoch,
        'completed_optimizer_updates': (
            len(fit.history) * steps_per_epoch
        ),
        'best_epoch': fit.best_epoch,
        'best_optimizer_update': (
            fit.best_epoch * steps_per_epoch
        ),
        'pair_presentations': (
            len(fit.history) * optimization_calibration_tokens
        ),
        'selected_train_mse': train_metrics.mse,
        'selected_validation_mse': validation_metrics.mse,
        'train_validation_gap': (
            validation_metrics.mse - train_metrics.mse
        ),
        'heldout_nmse': validation_metrics.relative_mse,
        'heldout_cosine': (
            validation_metrics.cosine_similarity
        ),
    })
    for epoch in fit.history:
        optimization_history_rows.append({
            'configuration': spec['configuration'],
            'epoch': epoch.epoch,
            'optimizer_updates': (
                epoch.epoch * steps_per_epoch
            ),
            'operator_batch_size': training_config.batch_size,
            'pair_presentations': (
                epoch.epoch * optimization_calibration_tokens
            ),
            'learning_rate': epoch.learning_rate,
            'train_mse': epoch.train_mse,
            'validation_mse': epoch.validation_mse,
        })

if RUN_FROM_SCRATCH:
    training_optimization_df = pd.DataFrame(optimization_rows)
    training_optimization_history_df = pd.DataFrame(
        optimization_history_rows
    )
    best_training_configuration = (
        training_optimization_df
        .sort_values('heldout_nmse')
        .iloc[0]['configuration']
    )
    best_training_spec = next(
        spec.copy() for spec in TRAINING_OPTIMIZATION_SPECS
        if spec['configuration'] == best_training_configuration
    )
else:
    optimization_results = loaded_artifact['results'][
        'training_optimization'
    ]
    training_optimization_df = pd.DataFrame(
        optimization_results['summary']
    )
    training_optimization_history_df = pd.DataFrame(
        optimization_results['training_history']
    )
    best_training_configuration = optimization_results[
        'best_configuration'
    ]
    best_training_spec = optimization_results['best_spec']

In [ ]:
training_optimization_report_df = (
    training_optimization_df
    .sort_values('heldout_nmse')
    .reset_index(drop=True)
)
display(training_optimization_report_df)
print(
    'Selected operator batch size: '
    f"{best_training_spec['operator_batch_size']:,}"
)

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
sns.lineplot(
    data=training_optimization_history_df,
    x='epoch',
    y='train_mse',
    hue='operator_batch_size',
    ax=axes[0],
)
sns.lineplot(
    data=training_optimization_history_df,
    x='epoch',
    y='validation_mse',
    hue='operator_batch_size',
    ax=axes[1],
)
axes[0].set(title='Batch-size screen: training', ylabel='MSE')
axes[1].set(title='Batch-size screen: validation', ylabel='MSE')
plt.tight_layout()
plt.show()

In [ ]:
plt.figure(figsize=(7, 4))
sns.lineplot(
    data=training_optimization_report_df,
    x='operator_batch_size',
    y='heldout_nmse',
    marker='o',
)
plt.xscale('log', base=2)
plt.xticks(OPERATOR_BATCH_SIZES, OPERATOR_BATCH_SIZES)
plt.title('Operator batch-size comparison')
plt.xlabel('operator batch size')
plt.ylabel('held-out NMSE')
plt.tight_layout()
plt.show()

#### Calibration data + training optimization

Repeat the 64-epoch calibration-size study with the batch size selected by the short screen, then compare it directly with the batch-2,048 study.

In [ ]:
combined_rows = []
combined_history_rows = []

for num_batches in (
    CALIBRATION_BATCH_BUDGETS if RUN_FROM_SCRATCH else []
):
    num_tokens = num_batches * tokens_per_capture_batch
    training_pairs = ActivationPairs(
        inputs=calibration_training_pool.inputs[:num_tokens],
        targets=calibration_training_pool.targets[:num_tokens],
    )
    steps_per_epoch = ceil(
        num_tokens / best_training_spec['operator_batch_size']
    )
    training_config = replace(
        baseline_training_config,
        epochs=CALIBRATION_MAX_EPOCHS,
        batch_size=best_training_spec['operator_batch_size'],
    )

    torch.manual_seed(SEED)
    fit = fit_operator(
        GatedMLPReplacement(
            hidden_size, replacement_width, bias=False
        ),
        training_pairs,
        calibration_validation_pairs,
        training_config,
        device,
    )
    train_metrics = evaluate_operator(
        fit.module,
        training_pairs,
        device,
        training_config.batch_size,
    )
    validation_metrics = evaluate_operator(
        fit.module,
        calibration_validation_pairs,
        device,
        training_config.batch_size,
    )

    combined_rows.append({
        'calibration_batches': num_batches,
        'calibration_tokens': num_tokens,
        'configuration': best_training_configuration,
        'operator_batch_size': training_config.batch_size,
        'maximum_epochs': training_config.epochs,
        'completed_epochs': len(fit.history),
        'steps_per_epoch': steps_per_epoch,
        'completed_optimizer_updates': (
            len(fit.history) * steps_per_epoch
        ),
        'best_epoch': fit.best_epoch,
        'best_optimizer_update': (
            fit.best_epoch * steps_per_epoch
        ),
        'pair_presentations': (
            len(fit.history) * num_tokens
        ),
        'selected_train_mse': train_metrics.mse,
        'selected_validation_mse': validation_metrics.mse,
        'train_validation_gap': (
            validation_metrics.mse - train_metrics.mse
        ),
        'heldout_nmse': validation_metrics.relative_mse,
        'heldout_cosine': (
            validation_metrics.cosine_similarity
        ),
    })
    for epoch in fit.history:
        combined_history_rows.append({
            'calibration_tokens': num_tokens,
            'epoch': epoch.epoch,
            'optimizer_updates': epoch.epoch * steps_per_epoch,
            'pair_presentations': epoch.epoch * num_tokens,
            'train_mse': epoch.train_mse,
            'validation_mse': epoch.validation_mse,
        })

if RUN_FROM_SCRATCH:
    combined_training_df = pd.DataFrame(combined_rows)
    combined_training_history_df = pd.DataFrame(
        combined_history_rows
    )
    combined_comparison_df = (
        calibration_study_df[
            [
                'calibration_tokens',
                'operator_batch_size',
                'completed_optimizer_updates',
                'heldout_nmse',
            ]
        ]
        .rename(columns={
            'operator_batch_size': 'baseline_batch_size',
            'completed_optimizer_updates': (
                'baseline_optimizer_updates'
            ),
            'heldout_nmse': 'original_nmse',
        })
        .merge(
            combined_training_df[
                [
                    'calibration_tokens',
                    'operator_batch_size',
                    'completed_optimizer_updates',
                    'heldout_nmse',
                    'heldout_cosine',
                    'train_validation_gap',
                ]
            ].rename(columns={
                'operator_batch_size': 'optimized_batch_size',
                'completed_optimizer_updates': (
                    'optimized_optimizer_updates'
                ),
                'heldout_nmse': 'optimized_nmse',
            }),
            on='calibration_tokens',
        )
    )
    combined_comparison_df['nmse_reduction'] = (
        combined_comparison_df['original_nmse']
        - combined_comparison_df['optimized_nmse']
    )
else:
    combined_results = loaded_artifact['results']['combined']
    combined_training_df = pd.DataFrame(
        combined_results['summary']
    )
    combined_training_history_df = pd.DataFrame(
        combined_results['training_history']
    )
    combined_comparison_df = pd.DataFrame(
        combined_results['comparison']
    )

In [ ]:
display(combined_comparison_df)

In [ ]:
combined_history_plot_df = combined_training_history_df.copy()
combined_history_plot_df['calibration budget'] = (
    combined_history_plot_df['calibration_tokens']
    .map(lambda value: f'{value:,}')
)

fig, axes = plt.subplots(1, 2, figsize=(12, 4), sharey=True)
sns.lineplot(
    data=combined_history_plot_df,
    x='epoch',
    y='train_mse',
    hue='calibration budget',
    ax=axes[0],
)
sns.lineplot(
    data=combined_history_plot_df,
    x='epoch',
    y='validation_mse',
    hue='calibration budget',
    ax=axes[1],
)
axes[0].set(title='Combined: online training MSE', ylabel='MSE')
axes[1].set(title='Combined: validation MSE', ylabel='MSE')
plt.tight_layout()
plt.show()

In [ ]:
combined_plot_df = combined_comparison_df.melt(
    id_vars='calibration_tokens',
    value_vars=['original_nmse', 'optimized_nmse'],
    var_name='training',
    value_name='heldout_nmse',
)
combined_plot_df['training'] = combined_plot_df['training'].map({
    'original_nmse': 'original',
    'optimized_nmse': best_training_configuration,
})

plt.figure(figsize=(7, 4))
sns.lineplot(
    data=combined_plot_df,
    x='calibration_tokens',
    y='heldout_nmse',
    hue='training',
    marker='o',
)
plt.title('Calibration data and training configuration')
plt.xlabel('calibration activation pairs')
plt.ylabel('held-out NMSE')
plt.tight_layout()
plt.show()

saving artifact

In [ ]:
def json_records(frame):
    return json.loads(
        frame.to_json(orient='records', double_precision=15)
    )


if RUN_FROM_SCRATCH:
    artifact = {
        'schema_version': ARTIFACT_SCHEMA_VERSION,
        'created_at_utc': datetime.now(timezone.utc).isoformat(),
        'configuration': {
            'model': {
                **asdict(model_config),
                'resolved_revision': getattr(
                    model.config, '_commit_hash', None
                ),
            },
            'data': asdict(data_config),
            'calibration_partition': {
                'original_calibration_batches': (
                    data_config.num_calibration_batches
                ),
                'operator_validation_batches': (
                    data_config.num_operator_validation_batches
                ),
                'additional_calibration_batches': (
                    additional_calibration_batches
                ),
                'total_calibration_batches': (
                    max(CALIBRATION_BATCH_BUDGETS)
                ),
            },
            'baseline_training': asdict(
                baseline_training_config
            ),
            'seed': SEED,
            'target_layer': TARGET_LAYER,
            'swiglu_width_ratio': SWIGLU_WIDTH_RATIO,
            'hidden_size': hidden_size,
            'original_intermediate_width': (
                original_intermediate_width
            ),
            'replacement_width': replacement_width,
            'original_calibration_tokens': calibration_tokens,
            'calibration_batch_budgets': list(
                CALIBRATION_BATCH_BUDGETS
            ),
            'calibration_max_epochs': CALIBRATION_MAX_EPOCHS,
            'operator_batch_sizes': list(OPERATOR_BATCH_SIZES),
            'training_optimization_epochs': (
                TRAINING_OPTIMIZATION_EPOCHS
            ),
            'training_optimization_specs': (
                TRAINING_OPTIMIZATION_SPECS
            ),
        },
        'results': {
            'original_distillation': {
                'configuration': json_records(
                    baseline_config_df
                ),
                'metrics': json_records(baseline_metrics_df),
                'training_history': json_records(
                    baseline_history_df
                ),
                'width_scaling': json_records(width_scaling_df),
            },
            'calibration_data': {
                'summary': json_records(calibration_study_df),
                'training_history': json_records(
                    calibration_history_df
                ),
            },
            'training_optimization': {
                'summary': json_records(
                    training_optimization_df
                ),
                'training_history': json_records(
                    training_optimization_history_df
                ),
                'best_configuration': (
                    best_training_configuration
                ),
                'best_spec': best_training_spec,
            },
            'combined': {
                'summary': json_records(combined_training_df),
                'training_history': json_records(
                    combined_training_history_df
                ),
                'comparison': json_records(
                    combined_comparison_df
                ),
            },
        },
    }
    OUTPUT_PATH.parent.mkdir(parents=True, exist_ok=True)
    OUTPUT_PATH.write_text(
        json.dumps(artifact, indent=2, allow_nan=False),
        encoding='utf-8',
    )
    print(f'Saved operator-distillation artifact to {OUTPUT_PATH}')
else:
    artifact = loaded_artifact
    print(f'Loaded operator-distillation artifact from {OUTPUT_PATH}')

## Findings & Solutions

The following section contains findings in study of operator fitting, documents problems and their respective solutions.

1. Calibration data and training budget

   Problem: The original study limited every data size to 384 optimizer updates. More calibration data therefore meant fewer passes over each pair, so the 0.487 NMSE result was a fixed-compute result rather than a clean data-scaling result.

   Solution: The reworked study gives every calibration size the same maximum of 64 epochs at batch size 2,048, with early stopping. Larger datasets now produce proportionally more optimizer updates.

2. Random student initialization

   Problem: The student starts with random weights and must relearn a large nonlinear operator only from input-output pairs. The planned comparison with teacher-derived initialization is not implemented.

   Solution: First verify that a full-width student initialized from the teacher gives near-zero NMSE. Then initialize reduced-width students by selecting teacher intermediate units and fine-tune them with local distillation.

3. Width is not the main limitation in the current setting

   Problem: Increasing the student width from 25% to 90% improves NMSE only from about 0.647 to 0.610. More capacity does not solve insufficient data or optimization when the student is trained from scratch.

   Solution: Compare random and teacher-derived initialization at the same widths before increasing model size further. Include a full-width random student to separate optimization failure from compression error.

4. The target may be too strict for the selected compression

   Problem: Layer 11 intermediate activations are not strongly low-dimensional. A non-deployable rank-4096 PCA reconstruction still has about 0.232 output NMSE. This is not a lower bound for SwiGLU, but it shows that retaining half the intermediate dimension does not preserve almost all output information automatically. The NMSE target of 0.05 is currently a project goal rather than an empirically justified threshold.

   Solution: Use the PCA result as an oracle reference, not as a direct baseline. Determine realistic local targets together with model-level quality measurements and test whether teacher-derived structured compression can outperform the PCA reference.

5. The original training optimization search was not informative

   Problem: Learning-rate, scheduler, and weight-decay variants produced nearly identical results, while the operator batch size remained fixed at 2,048. That large batch gives relatively few parameter updates per epoch.

   Solution: The reworked optimization section compares batch sizes 256, 512, 1,024, and 2,048 on the largest calibration pool for 8 complete epochs. The selected batch size is then used in the combined 64-epoch calibration study.

6. The historical baseline used the wrong width semantics

   Problem: The copied historical result interpreted a 0.50 ratio against the 2,048-wide model representation, producing a 1,024-wide operator with 6.29 million parameters. A true 0.50 reduction of the teacher d_ff=8,192 requires a 4,096-wide operator with 25.17 million parameters.

   Solution: The notebook now recomputes the corrected baseline directly. It uses the original 48 calibration batches and a fixed 24-batch validation partition before drawing any additional calibration data, so every later comparison reuses the same baseline data and validation set.

7. The train-validation gap is not measured consistently

   Problem: The epoch training MSE is averaged while the model changes, whereas validation MSE is measured using one frozen state. Subtracting those values did not give an exact generalization gap.

   Solution: The summary tables now recompute training and validation MSE from the selected checkpoint. History plots keep the online training MSE and label it accordingly.

8. The validation set is reused for several decisions

   Problem: The same operator-validation set selects the best epoch, calibration budget, and training configuration. It is document-disjoint from training, which is good, but repeated selection makes it unsuitable as an unbiased final local evaluation.

   Solution: Keep this split for model selection and introduce a separate operator-test split that is evaluated only after the fitting recipe is frozen.

9. Local fit is not final model quality

   Problem: NMSE measures isolated MLP imitation and does not directly determine language-model quality because residual connections and later layers can compensate for local errors. The best 0.487 operator is not evaluated inside the complete model in this notebook.

   Solution: Use NMSE, cosine similarity, R2, norm ratio, and token-level errors as local diagnostics. Make final decisions using matched full-model loss, perplexity, downstream quality, and compression measurements.